In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
!pip install datasets
from datasets import load_dataset

dataset = load_dataset("imdb")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
dataset["train"]
dataset["test"]

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [ ]:
import re

def tokenize(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z]", " ", text)
    tokens = text.split()
    return tokens

In [ ]:
print(tokenize(dataset["train"][0]["text"])[:20])

['i', 'rented', 'i', 'am', 'curious', 'yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it']


In [ ]:
from collections import Counter

counter = Counter()

for sample in dataset["train"]:
    tokens = tokenize(sample["text"])
    counter.update(tokens)

In [ ]:
MAX_VOCAB_SIZE = 20000
MAX_LEN = 200

# Build vocabulary
most_common = counter.most_common(MAX_VOCAB_SIZE)

vocab = {"<PAD>": 0, "<UNK>": 1}

for idx, (word, _) in enumerate(most_common):
    vocab[word] = idx + 2

vocab_size = len(vocab)
print("Vocab size:", vocab_size)

# Encoding function
def encode(text):
    tokens = tokenize(text)

    ids = [vocab.get(word, vocab["<UNK>"]) for word in tokens]

    if len(ids) > MAX_LEN:
        ids = ids[:MAX_LEN]
    else:
        ids += [vocab["<PAD>"]] * (MAX_LEN - len(ids))

    return torch.tensor(ids, dtype=torch.long)

Vocab size: 20002


In [ ]:
vocab_size = len(vocab)
print(vocab_size)

20002


In [ ]:
class IMDBDataset(Dataset):

    def __init__(self, split):
        self.texts = split["text"]
        self.labels = split["label"]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = encode(self.texts[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.float)
        return text, label

In [ ]:
full_train_dataset = IMDBDataset(dataset["train"])
test_dataset = IMDBDataset(dataset["test"])

In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size]
)

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
batch = next(iter(train_loader))
print(batch[0].shape)
print(batch[1].shape)

torch.Size([64, 200])
torch.Size([64])


In [ ]:
class LSTMClassifier(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout=0.5):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Now bidirectional
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=0.2,          # internal dropout
            bidirectional=True    # added
        )

        # Tunable dropout
        self.dropout = nn.Dropout(dropout)

        # hidden_dim * 2 because bidirectional
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        embedded = self.embedding(x)

        output, (hidden, cell) = self.lstm(embedded)

        # hidden shape:
        # (num_layers * num_directions, batch_size, hidden_dim)
        # For 2 layers + bidirectional → (4, batch_size, hidden_dim)

        # We take last layer's forward and backward hidden states
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        # Concatenate them
        final_hidden = torch.cat((forward_hidden, backward_hidden), dim=1)

        dropped = self.dropout(final_hidden)
        logits = self.fc(dropped)

        return logits

In [ ]:
def binary_accuracy(preds, labels):
    probs = torch.sigmoid(preds)
    rounded = torch.round(probs)
    correct = (rounded == labels).float()
    return correct.sum() / len(correct)

In [ ]:
def train(model, loader, optimizer, criterion):

    model.train()
    total_loss = 0
    total_acc = 0

    for texts, labels in loader:

        texts = texts.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(texts).squeeze(1)

        loss = criterion(outputs, labels)
        acc = binary_accuracy(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_acc += acc.item()

    return total_loss / len(loader), total_acc / len(loader)

In [ ]:
def evaluate(model, loader, criterion):

    model.eval()
    total_loss = 0
    total_acc = 0

    with torch.no_grad():
        for texts, labels in loader:

            texts = texts.to(device)
            labels = labels.to(device)

            outputs = model(texts).squeeze(1)

            loss = criterion(outputs, labels)
            acc = binary_accuracy(outputs, labels)

            total_loss += loss.item()
            total_acc += acc.item()

    return total_loss / len(loader), total_acc / len(loader)

In [ ]:
!pip install optuna
import optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 11.5 MB/s eta 0:00:00


In [ ]:
def objective(trial):

    hidden_dim = trial.suggest_categorical("hidden_dim", [128, 256, 512])
    dropout = trial.suggest_float("dropout", 0.2, 0.6)
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)

    model = LSTMClassifier(
        vocab_size,
        embed_dim=128,
        hidden_dim=hidden_dim,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    EPOCHS = 5

    for epoch in range(EPOCHS):

        train_loss, train_acc = train(model, train_loader, optimizer, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        print(f"Trial {trial.number} | Epoch {epoch+1} | Val Acc: {val_acc:.4f}")

        # Report intermediate result to Optuna
        trial.report(val_acc, step=epoch)

        # Prune if needed
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return val_acc

In [ ]:
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner()
)

study.optimize(objective, n_trials=10)

print("Best trial:")
print(study.best_trial.params)
print("Best validation accuracy:", study.best_trial.value)

[I 2026-03-01 14:24:59,252] A new study created in memory with name: no-name-cf2dc494-c953-48bf-869b-febf6bcf4377


Trial 0 | Epoch 1 | Val Acc: 0.6371
Trial 0 | Epoch 2 | Val Acc: 0.7002
Trial 0 | Epoch 3 | Val Acc: 0.7004
Trial 0 | Epoch 4 | Val Acc: 0.7318


[I 2026-03-01 14:33:11,051] Trial 0 finished with value: 0.7592958860759493 and parameters: {'hidden_dim': 512, 'dropout': 0.2034670939309709, 'lr': 0.00017760568791146608}. Best is trial 0 with value: 0.7592958860759493.


Trial 0 | Epoch 5 | Val Acc: 0.7593
Trial 1 | Epoch 1 | Val Acc: 0.6216
Trial 1 | Epoch 2 | Val Acc: 0.7358
Trial 1 | Epoch 3 | Val Acc: 0.7601
Trial 1 | Epoch 4 | Val Acc: 0.7694


[I 2026-03-01 14:41:20,838] Trial 1 finished with value: 0.7913370253164557 and parameters: {'hidden_dim': 512, 'dropout': 0.5086594095593037, 'lr': 0.0004160112699577479}. Best is trial 1 with value: 0.7913370253164557.


...

Trial 7 | Epoch 2 | Val Acc: 0.7213


[I 2026-03-01 15:05:14,295] Trial 8 pruned. 


Trial 8 | Epoch 1 | Val Acc: 0.5528


[I 2026-03-01 15:06:52,469] Trial 9 pruned. 


Trial 9 | Epoch 1 | Val Acc: 0.5034
Best trial:
{'hidden_dim': 512, 'dropout': 0.5531047091907322, 'lr': 0.002170736443941711}
Best validation accuracy: 0.853243670886076


In [ ]:
best_params = study.best_trial.params

In [ ]:
best_hidden = 512
best_dropout = 0.5531047091907322
best_lr = 0.002170736443941711

model = LSTMClassifier(
    vocab_size,
    embed_dim=128,
    hidden_dim=best_hidden,
    dropout=best_dropout
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=best_lr)
criterion = nn.BCEWithLogitsLoss()

In [ ]:
EPOCHS = 10
best_val = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    print(f"Epoch {epoch+1} | Val Acc: {val_acc:.4f}")


    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), "best_bilstm_model.pt")
        print("Saved best model!")

print("Best Validation Accuracy:", best_val)

Epoch 1 | Val Acc: 0.8641
Saved best model!
Epoch 2 | Val Acc: 0.8580
Epoch 3 | Val Acc: 0.8521
Epoch 4 | Val Acc: 0.8509
Epoch 5 | Val Acc: 0.8625
Epoch 6 | Val Acc: 0.8560
Epoch 7 | Val Acc: 0.8554
Epoch 8 | Val Acc: 0.8572
Epoch 9 | Val Acc: 0.8574
Epoch 10 | Val Acc: 0.8600
Best Validation Accuracy: 0.864121835443038


In [ ]:
model.load_state_dict(torch.load("best_bilstm_model.pt"))
model.to(device)

LSTMClassifier(
  (embedding): Embedding(20002, 128, padding_idx=0)
  (lstm): LSTM(128, 512, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
  (dropout): Dropout(p=0.5531047091907322, inplace=False)
  (fc): Linear(in_features=1024, out_features=1, bias=True)
)

In [ ]:
from google.colab import files
files.download("best_bilstm_model.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion)
print("Test Accuracy:", test_acc)

Test Accuracy: 0.8390744884910486
